# Hidden Categories

![Grafika do zadania](https://i.imgur.com/7o3Y4WP.png)

*Source: Image generated using ChatGPT.*

## Introduction

Kuba has just started working at a company that runs an e-commerce store. One of his tasks is to describe product categories. Some products were already labeled by his predecessor, and the company cares a lot about consistent labeling. Unfortunately, no instructions were left behind. Help Kuba complete the missing labels.

Modern recommender systems often use a *collaborative filtering* approach, which learns solely from user behavior (e.g. clicks or purchases) — without using product information. Products that appear in similar contexts receive similar numerical representations (embeddings), reflecting their relationships.

## Task

Your task is to implement and train a model that, based on product embeddings coming from a recommendation system, predicts a **set of categories** assigned to each product. This set should ideally contain all original labels (not necessarily in the original order), without additional elements.

## Data

The data consists of embeddings (256-dimensional vectors representing products in a latent space) and a collection of product labels (lists of labels). It has been split into:

* training set (12,145 products),

* validation set (2,603 products),

* hidden test set (2,603 products).

Each row in the embedding matrix corresponds to the same product as the corresponding element in the list of label sets.

## Evaluation Metric

Your solution will be evaluated using **Intersection over Union (IoU)**, computed for each product:

$$IoU = \frac{|\text{prediction} \cap \text{ground truth}|}{|\text{prediction} \cup \text{ground truth}|}$$

The metric is calculated without considering the structure or order of labels.

The final score is the average IoU across all products, converted into points:

* if the result is **below 36%**, you will receive **0 points**,

* if the result is **above 46%**, you will receive the **maximum score of 100 points**.

Scores between these thresholds are scaled proportionally.

## Constraints

* You may only use the training set to train your model.
* Your solution will be evaluated on the Competition Platform without internet access and **without GPU**.
* Training and evaluation of your final solution must not exceed 5 minutes on the Competition Platform.
* Allowed libraries: `numpy`, `pandas`, `sklearn`.

## Submission Files

The solution to this task is this notebook completed with your implementation in the form of a `YourSolution` class (see section: ***Your Solution***).

## Tips

* Pay attention to the structure of labels. Each product may belong to **more than one category**.
* Product embeddings are optimized so that the higher the dot product between a user embedding and a product embedding, the higher the product is ranked in a personalized list.

## Evaluation

Remember that during evaluation, the flag `FINAL_EVALUATION_MODE` will be set to `True`.

You can score between 0 and 100 points. Your score, computed on a hidden test set on the Competition Platform using the formula above, will be rounded to an integer. If your solution does not meet the criteria or fails to run correctly, you will receive 0 points.


# Starter Code

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
import json
import numpy as np
import pandas as pd
import sklearn
import math

In [ ]:
FINAL_EVALUATION_MODE = False

## Data Loading

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
with open('data/train_categories.json', 'r') as f:
    train_categories = json.load(f)

train_embeddings = np.load('data/train_embeddings.npy', allow_pickle=True)

with open('data/val_categories.json', 'r') as f:
    val_categories = json.load(f)

val_embeddings = np.load('data/val_embeddings.npy', allow_pickle=True)

## Evaluation Metric Code

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def intersection_over_union(x, y):
    set_x = set(x)
    set_y = set(y)
    return len(set_x.intersection(set_y)) / len(set_x.union(set_y))

In [ ]:
def loss(true_sets, predicted_sets):
    return np.mean([intersection_over_union(x, y) for x, y in zip(true_sets, predicted_sets)])

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
def round_half_up(number: float) -> int:
    return int(math.floor(number + 0.5))

def compute_score(loss_val: float):
    upper_limit = 0.46
    lower_limit = 0.36
    if loss_val > upper_limit:
        return 100
    elif loss_val < lower_limit:
        return 0
    else:
        return round_half_up((loss_val - lower_limit) / (upper_limit - lower_limit) * 100)

# Your Solution

The solution must implement a class `YourSolution` that satisfies certain formal requirements described below.
The definition of the `YourSolution` class **must** implement the following methods:

1. A `fit` method that takes as an argument a `np.ndarray`, whose rows are 256-dimensional product embeddings, and a list whose elements are lists of labels for consecutive products.

2. A `predict` method that takes as an argument a `np.ndarray`, whose rows are 256-dimensional product embeddings. It should return a list whose elements are lists of labels for consecutive products.

This section contains an example solution. It is a very simple baseline based on the most frequent label. Thanks to this, the notebook runs correctly even without editing this cell.

You are free to modify the definition of this class as long as the above requirements are satisfied. You may add necessary attributes and methods and modify existing ones. Your solution will be evaluated **only** based on the `YourSolution` class.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

class YourSolution:
    def __init__(self):
        """
        __init__ should use hardcoded or default hyperparameter values
        """
        pass

    def fit(self, X, y):
        # Here you can include your data preprocessing
        # Train the model
        pass

    def predict(self, X):
        return [['Sports & Outdoors'] for _ in X]


# Evaluation

Running the cell below allows you to check how many points your solution would achieve on the training data. Before submission, make sure that the entire notebook runs from start to finish without errors and without requiring any user intervention after selecting "Run All".


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################
if not FINAL_EVALUATION_MODE:
    # Model training
    yourSolutionInstance = YourSolution()
    yourSolutionInstance.fit(train_embeddings, train_categories)

    # Prediction preparation
    predictions = yourSolutionInstance.predict(val_embeddings)

    # Solution evaluation
    loss_val = loss(val_categories, predictions)
    print(compute_score(loss_val))